# Music Annotation Project
## Notebook 77: Observable-Paper Veto for Ring-2 Preservation

**Runtime:** CPU. Select **Runtime → Run all**.

Notebook 77 tests a narrow routing hypothesis before C1 is adopted:

> Preserve the C1 Ring-2 restoration by default, including unresolved and externally print-supported regions, but revert to the C0 incumbent where the annotated page itself provides positive observable evidence of paper and no competing print-support evidence.

This is deliberately a **paper veto**, not another deterministic positive-print selector. Earlier deterministic work showed that geometry was more useful as exclusion evidence than as sufficient proof of print. Notebook 77 therefore asks whether observable paper can remove C1's uninformative annotation return without throwing away its small whole-page benefit.

## Arms

| Arm | Treatment |
|---|---|
| `C0_INCUMBENT` | Notebook 75 incumbent everywhere |
| `C1_RING2` | Notebook 75 Ring-2 preservation everywhere |
| `A1_PRINT_SUPPORTED_ONLY` | C0 except C1 candidate pixels with positive external print support |
| `A2_PAPER_VETO` | C1 except candidate pixels with positive paper evidence and no print support |
| `A3_PAPER_VETO_PLUS_RING1_STRONG` | A2 plus only C2 Ring-1 pixels with strong multi-channel external support |
| `ORACLE_PAPER_VETO` | Development ceiling using clean truth after prediction lock only |

`A2_PAPER_VETO` is the primary observable challenger. `A1` and `A3` are diagnostic comparators. `ORACLE_PAPER_VETO` is not deployable and exists only to show the opportunity ceiling.

## Observable-only inference evidence

- local paper surface estimated from pixels outside the authoritative mask;
- outside-mask dark-ink evidence;
- horizontal, vertical and compact support derived from outside-mask dark material;
- fixed observed edge evidence;
- distance to outside dark/edge/support;
- local outside-paper sample density;
- mask depth and C1/C2 candidate identity.

Clean truth is loaded only **after** all observable maps, decisions and output pages are written and hashed.

## Interpretation

A positive result means observable paper is a useful veto for C1. It does not prove that every retained unresolved pixel is genuine print. A negative result means the observable-paper definition is either insufficiently selective or removes too much useful C1 recovery.

Notebook 77 does not modify the incumbent, Notebook 75, `v8_8_operator.py`, or production outputs.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib seaborn pyarrow

## 1. Configuration and predeclared observable-paper policy

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, zipfile, sys, platform
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
NB75_ROOT=ROOT/'notebook75_v88_collar_preservation_counterfactual'
NB75A_ROOT=ROOT/'notebook75a_stratified_reanalysis'
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
OUTPUT_ROOT=ROOT/'notebook77_observable_paper_veto'
SOURCE_NB75_RUN_OVERRIDE=None
SOURCE_NB75A_RUN_OVERRIDE=None
SOURCE_V75_RUN_OVERRIDE=None
PAGE_ALLOWLIST=None

# Frozen evidence thresholds carried from the established geometry lineage.
PAPER_SAMPLE_MIN=200
OUTSIDE_DARK_THRESHOLD=190
OBSERVED_DARK_THRESHOLD=200
EDGE_LOW=70
EDGE_HIGH=170
BACKGROUND_SIGMA=45.0
SUPPORT_BANDS=(2,4,8,16)

# Primary A2 paper-veto rule. These values are declared before scoring.
PRIMARY_SUPPORT_BAND=8
PRIMARY_PAPER_RADIUS=12
PRIMARY_MIN_PAPER_SAMPLE_FRACTION=0.72
PRIMARY_MAX_OUTSIDE_DARK_FRACTION=0.035
PRIMARY_MAX_OUTSIDE_EDGE_FRACTION=0.055
PRIMARY_MIN_NORMALISED_INTENSITY=0.72
PRIMARY_MIN_DISTANCE_TO_DARK=4.0
PRIMARY_MIN_DISTANCE_TO_EDGE=3.0

# A3 is intentionally narrow: Ring 1 is restored only with multi-channel support.
A3_MIN_SUPPORT_CHANNELS=2
A3_MAX_MASK_DEPTH=2.5

ORACLE_NEAR_PRINT_RADII=(0,1,2,4,8,16)
BOOTSTRAP_RESAMPLES=10000
SEED=42

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook77_observable_paper_veto'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; MAPS=OUT/'observable_maps'; PAGES=OUT/'pages'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; VISUAL=OUT/'visuals'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,MAPS,PAGES,TABLES,AUDIT,VISUAL,UPLOAD]:p.mkdir(parents=True,exist_ok=False)

def require(x,msg):
    if not bool(x): raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a);h=hashlib.sha256();h.update(str(a.dtype).encode());h.update(str(a.shape).encode());h.update(a.tobytes());return h.hexdigest()
def object_hash(x):return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def atomic_json(path,x):
    t=Path(str(path)+'.tmp');t.write_text(json.dumps(x,indent=2,sort_keys=True,default=str));t.replace(path)
def latest(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

POLICY={
 'support_bands':SUPPORT_BANDS,'primary_support_band':PRIMARY_SUPPORT_BAND,'primary_paper_radius':PRIMARY_PAPER_RADIUS,
 'paper_sample_min':PAPER_SAMPLE_MIN,'outside_dark_threshold':OUTSIDE_DARK_THRESHOLD,'observed_dark_threshold':OBSERVED_DARK_THRESHOLD,
 'minimum_paper_sample_fraction':PRIMARY_MIN_PAPER_SAMPLE_FRACTION,'maximum_outside_dark_fraction':PRIMARY_MAX_OUTSIDE_DARK_FRACTION,
 'maximum_outside_edge_fraction':PRIMARY_MAX_OUTSIDE_EDGE_FRACTION,'minimum_normalised_intensity':PRIMARY_MIN_NORMALISED_INTENSITY,
 'minimum_distance_to_dark':PRIMARY_MIN_DISTANCE_TO_DARK,'minimum_distance_to_edge':PRIMARY_MIN_DISTANCE_TO_EDGE,
 'a3_min_support_channels':A3_MIN_SUPPORT_CHANNELS,'a3_max_mask_depth':A3_MAX_MASK_DEPTH,
}
atomic_json(INPUTS/'predeclared_policy.json',POLICY)
print('Notebook 77 output:',OUT)

## 2. Resolve completed Notebook 75 / 75A and the canonical V7.5 cache

In [ ]:
NB75=Path(SOURCE_NB75_RUN_OVERRIDE) if SOURCE_NB75_RUN_OVERRIDE else latest(NB75_ROOT,['tables/results_by_page.csv','pages','prediction_lock/prediction_lock.json'])
NB75A=Path(SOURCE_NB75A_RUN_OVERRIDE) if SOURCE_NB75A_RUN_OVERRIDE else latest(NB75A_ROOT,['tables/page_role_resolution.csv','tables/collar_exposed_arm_summary.csv','notebook75a_verdict.json'])
V75=Path(SOURCE_V75_RUN_OVERRIDE) if SOURCE_V75_RUN_OVERRIDE else latest(V75_ROOT,['tables/cache_status.csv'])
require(NB75 is not None,'No complete Notebook 75 run found.')
require(NB75A is not None,'No complete Notebook 75A run found.')
require(V75 is not None,'No complete V7.5 cache run found.')

results75=pd.read_csv(NB75/'tables'/'results_by_page.csv')
roles=pd.read_csv(NB75A/'tables'/'page_role_resolution.csv')
status=pd.read_csv(V75/'tables'/'cache_status.csv')
cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
exposed=roles[roles.c2_restored_pixels>0].copy()
if PAGE_ALLOWLIST is not None: exposed=exposed[exposed.page_key.isin(set(PAGE_ALLOWLIST))]
require(len(exposed)>0,'No collar-exposed pages selected.')

manifest_path=ROOT/'development_score_manifest.csv'
manifest=pd.read_csv(manifest_path,dtype=str).fillna('') if manifest_path.exists() else pd.DataFrame()
if len(manifest) and {'page','source_group'}.issubset(manifest.columns):
    metadata=manifest[['page','source_group']+[c for c in ['acquisition_notes'] if c in manifest.columns]].drop_duplicates('page')
else:
    metadata=exposed[['page','source_group']].drop_duplicates()

source_record={'notebook75':str(NB75),'notebook75a':str(NB75A),'v75_cache':str(V75),'notebook75_prediction_lock_sha256':sha256_file(NB75/'prediction_lock'/'prediction_lock.json'),'notebook75_results_sha256':sha256_file(NB75/'tables'/'results_by_page.csv'),'notebook75a_roles_sha256':sha256_file(NB75A/'tables'/'page_role_resolution.csv'),'predeclared_policy_sha256':sha256_file(INPUTS/'predeclared_policy.json')}
atomic_json(AUDIT/'source_record.json',source_record)
print('Notebook 75:',NB75);print('Notebook 75A:',NB75A);print('V7.5:',V75);print('Exposed pages:',len(exposed))

## 3. Image/array resolution and observable evidence helpers

In [ ]:
IMAGE_NAMES={
 'observed':['observed.png','OBSERVED.png'],
 'C0':['C0_INCUMBENT.png','c0_incumbent.png'],
 'C1':['C1_PRESERVE_RING2.png','c1_preserve_ring2.png'],
 'C2':['C2_PRESERVE_RING1_RING2.png','c2_preserve_ring1_ring2.png'],
}
def resolve_image(folder,names):
    for n in names:
        p=folder/n
        if p.exists():return p
    return None

def load_page_images(page_key):
    d=NB75/'pages'/page_key
    require(d.exists(),f'{page_key}: Notebook 75 page directory missing')
    out={}
    for k,names in IMAGE_NAMES.items():
        p=resolve_image(d,names);require(p is not None,f'{page_key}: missing {k} image in {d}')
        im=cv2.imread(str(p),0);require(im is not None,f'{page_key}: unreadable {p}')
        out[k]=im
    require(len({x.shape for x in out.values()})==1,f'{page_key}: shape mismatch among Notebook 75 images')
    return out

def background_surface(observed,mask,sigma=BACKGROUND_SIGMA):
    visible=(~mask)&(observed>=PAPER_SAMPLE_MIN)
    num=cv2.GaussianBlur(observed.astype(np.float32)*visible.astype(np.float32),(0,0),sigma)
    den=cv2.GaussianBlur(visible.astype(np.float32),(0,0),sigma)
    fallback=cv2.GaussianBlur(observed,(0,0),sigma)
    return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)

def disk(radius):return cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*radius+1,2*radius+1))
def local_fraction(binary,radius):
    k=disk(radius).astype(np.float32);return cv2.filter2D(binary.astype(np.float32),-1,k,borderType=cv2.BORDER_REFLECT)/max(float(k.sum()),1)
def distance_to(binary):
    # distanceTransform measures distance to zero, hence invert target map.
    return cv2.distanceTransform((~binary).astype(np.uint8),cv2.DIST_L2,5)
def support_base(observed,mask):
    dark=(observed<OUTSIDE_DARK_THRESHOLD)&~mask
    horizontal=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    vertical=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
    n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);compact=np.zeros_like(dark)
    for i in range(1,n):
        x,y,w,h,a=stats[i]
        if 3<=a<=650 and w<=40 and h<=40:compact[lab==i]=True
    return dark,horizontal,vertical,compact

def observable_context(observed,mask):
    paper=background_surface(observed,mask);normalised=observed.astype(np.float32)/np.maximum(paper.astype(np.float32),1)
    dark,h0,v0,c0=support_base(observed,mask);edge=(cv2.Canny(observed,EDGE_LOW,EDGE_HIGH)>0)&~mask
    supports={}
    for band in SUPPORT_BANDS:
        k=disk(band);supports[band]={
          'horizontal':cv2.dilate(h0.astype(np.uint8),k)>0,
          'vertical':cv2.dilate(v0.astype(np.uint8),k)>0,
          'compact':cv2.dilate(c0.astype(np.uint8),k)>0,
          'edge':cv2.dilate(edge.astype(np.uint8),k)>0,
        }
    outside_paper=(~mask)&(observed>=PAPER_SAMPLE_MIN)
    return {
      'paper':paper,'normalised':normalised,'outside_dark':dark,'outside_edge':edge,
      'paper_fraction':local_fraction(outside_paper,PRIMARY_PAPER_RADIUS),
      'dark_fraction':local_fraction(dark,PRIMARY_PAPER_RADIUS),
      'edge_fraction':local_fraction(edge,PRIMARY_PAPER_RADIUS),
      'distance_to_dark':distance_to(dark),'distance_to_edge':distance_to(edge),
      'mask_depth':cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5),
      'supports':supports,
    }

## 4. Build observable A1/A2/A3 arms before clean truth is loaded

The primary A2 rule requires **positive paper evidence** and no print support. Anything unresolved remains C1.

In [ ]:
page_runtime={}; decision_rows=[]; integrity_rows=[]
for _,rr in exposed.sort_values('page_key').iterrows():
    page_key=str(rr.page_key);page=str(rr.page);group=str(rr.source_group);imgs=load_page_images(page_key)
    require(page in cache,f'{page_key}: no V7.5 cache for {page}')
    with np.load(cache[page]) as z:
        require('mask' in z.files,f'{page}: cache has no prepared mask')
        mask=z['mask']>0
    require(mask.shape==imgs['observed'].shape,f'{page_key}: mask/image shape mismatch')
    observed,c0,c1,c2=imgs['observed'],imgs['C0'],imgs['C1'],imgs['C2']
    # Candidate identity is measured from completed Notebook 75 pages, so no arm reconstruction is guessed.
    ring2_candidate=(c1!=c0)&(c1==observed)&mask
    ring1_candidate=(c2!=c1)&(c2==observed)&mask
    ctx=observable_context(observed,mask);sp=ctx['supports'][PRIMARY_SUPPORT_BAND]
    support_count=sp['horizontal'].astype(np.uint8)+sp['vertical'].astype(np.uint8)+sp['compact'].astype(np.uint8)
    print_supported=(support_count>=1)|sp['edge']
    paper_supported=(
        ring2_candidate
        &(ctx['paper_fraction']>=PRIMARY_MIN_PAPER_SAMPLE_FRACTION)
        &(ctx['dark_fraction']<=PRIMARY_MAX_OUTSIDE_DARK_FRACTION)
        &(ctx['edge_fraction']<=PRIMARY_MAX_OUTSIDE_EDGE_FRACTION)
        &(ctx['normalised']>=PRIMARY_MIN_NORMALISED_INTENSITY)
        &(ctx['distance_to_dark']>=PRIMARY_MIN_DISTANCE_TO_DARK)
        &(ctx['distance_to_edge']>=PRIMARY_MIN_DISTANCE_TO_EDGE)
        &~print_supported
    )
    unresolved=ring2_candidate&~paper_supported&~print_supported
    a1=c0.copy();a1[ring2_candidate&print_supported]=observed[ring2_candidate&print_supported]
    a2=c1.copy();a2[paper_supported]=c0[paper_supported]
    strong_ring1=ring1_candidate&(support_count>=A3_MIN_SUPPORT_CHANNELS)&(ctx['mask_depth']<=A3_MAX_MASK_DEPTH)&(observed<OBSERVED_DARK_THRESHOLD)
    a3=a2.copy();a3[strong_ring1]=observed[strong_ring1]
    # Direct equivalence and immutability checks.
    require(np.array_equal(a1[~mask],observed[~mask]),f'{page_key}: A1 outside-mask drift')
    require(np.array_equal(a2[~mask],observed[~mask]),f'{page_key}: A2 outside-mask drift')
    require(np.array_equal(a3[~mask],observed[~mask]),f'{page_key}: A3 outside-mask drift')
    require(np.array_equal(a2[~paper_supported],c1[~paper_supported]),f'{page_key}: A2 changed non-veto pixels')
    require(np.array_equal(a2[paper_supported],c0[paper_supported]),f'{page_key}: A2 veto does not equal C0')
    d=PAGES/page_key;d.mkdir(parents=True,exist_ok=False)
    for name,im in [('observed',observed),('C0_INCUMBENT',c0),('C1_RING2',c1),('C2_RING1_RING2',c2),('A1_PRINT_SUPPORTED_ONLY',a1),('A2_PAPER_VETO',a2),('A3_PAPER_VETO_PLUS_RING1_STRONG',a3)]:cv2.imwrite(str(d/f'{name}.png'),im)
    # Visual decision map: blue print-supported, white paper-veto, amber unresolved, magenta strong ring1.
    overlay=cv2.cvtColor(observed,cv2.COLOR_GRAY2BGR);overlay[print_supported&ring2_candidate]=(255,80,20);overlay[paper_supported]=(255,255,255);overlay[unresolved]=(0,191,255);overlay[strong_ring1]=(255,0,255);cv2.imwrite(str(d/'observable_context_overlay.png'),overlay)
    np.savez_compressed(MAPS/f'{page_key}.npz',mask=mask.astype(np.uint8),ring2_candidate=ring2_candidate.astype(np.uint8),ring1_candidate=ring1_candidate.astype(np.uint8),paper_supported=paper_supported.astype(np.uint8),print_supported=print_supported.astype(np.uint8),unresolved=unresolved.astype(np.uint8),strong_ring1=strong_ring1.astype(np.uint8),paper=ctx['paper'],normalised=ctx['normalised'].astype(np.float32),paper_fraction=ctx['paper_fraction'].astype(np.float32),dark_fraction=ctx['dark_fraction'].astype(np.float32),edge_fraction=ctx['edge_fraction'].astype(np.float32),distance_to_dark=ctx['distance_to_dark'].astype(np.float32),distance_to_edge=ctx['distance_to_edge'].astype(np.float32),mask_depth=ctx['mask_depth'].astype(np.float32),support_count=support_count.astype(np.uint8))
    page_runtime[page_key]={'page':page,'source_group':group,'observed':observed,'mask':mask,'C0':c0,'C1':c1,'C2':c2,'A1':a1,'A2':a2,'A3':a3,'ring2':ring2_candidate,'ring1':ring1_candidate,'paper':paper_supported,'print_support':print_supported,'unresolved':unresolved,'strong_ring1':strong_ring1}
    integrity_rows.append({'page_key':page_key,'page':page,'source_group':group,'ring2_candidate_pixels':int(ring2_candidate.sum()),'paper_veto_pixels':int(paper_supported.sum()),'print_supported_pixels':int((ring2_candidate&print_supported).sum()),'unresolved_pixels':int(unresolved.sum()),'ring1_candidate_pixels':int(ring1_candidate.sum()),'a3_strong_ring1_pixels':int(strong_ring1.sum()),'A1_outside_drift':int(np.sum(a1[~mask]!=observed[~mask])),'A2_outside_drift':int(np.sum(a2[~mask]!=observed[~mask])),'A3_outside_drift':int(np.sum(a3[~mask]!=observed[~mask])),'A2_non_veto_regression':int(np.sum(a2[~paper_supported]!=c1[~paper_supported])),'observable_map_sha256':sha256_file(MAPS/f'{page_key}.npz')})
    for cls,m in [('PAPER_SUPPORTED',paper_supported),('PRINT_SUPPORTED',ring2_candidate&print_supported),('UNRESOLVED',unresolved),('A3_STRONG_RING1',strong_ring1)]:
        decision_rows.append({'page_key':page_key,'page':page,'source_group':group,'context_class':cls,'pixels':int(m.sum())})

integrity=pd.DataFrame(integrity_rows);integrity.to_csv(AUDIT/'observable_arm_integrity.csv',index=False);pd.DataFrame(decision_rows).to_csv(TABLES/'observable_context_counts.csv',index=False)
require(integrity[['A1_outside_drift','A2_outside_drift','A3_outside_drift','A2_non_veto_regression']].to_numpy().sum()==0,'Observable arm integrity failed.')
print(integrity[['ring2_candidate_pixels','paper_veto_pixels','print_supported_pixels','unresolved_pixels','a3_strong_ring1_pixels']].sum())

## 5. Prediction/output lock before clean truth

In [ ]:
page_hashes=[]
for page_key,z in page_runtime.items():
    row={'page_key':page_key,'page':z['page'],'source_group':z['source_group']}
    for k in ['observed','C0','C1','C2','A1','A2','A3']:row[k+'_array_sha256']=array_hash(z[k])
    for k in ['ring2','ring1','paper','print_support','unresolved','strong_ring1']:row[k+'_array_sha256']=array_hash(z[k].astype(np.uint8))
    page_hashes.append(row)
pd.DataFrame(page_hashes).to_csv(AUDIT/'page_output_hashes.csv',index=False)
lock_payload={'created_utc':datetime.now(timezone.utc).isoformat(),'source_record':source_record,'policy':POLICY,'pages':len(page_runtime),'page_hashes_sha256':sha256_file(AUDIT/'page_output_hashes.csv'),'integrity_sha256':sha256_file(AUDIT/'observable_arm_integrity.csv'),'context_counts_sha256':sha256_file(TABLES/'observable_context_counts.csv')}
prediction_lock={'lock_id':object_hash(lock_payload),'state':'OBSERVABLE_DECISIONS_LOCKED_BEFORE_TRUTH','payload':lock_payload}
atomic_json(OUT/'prediction_lock.json',prediction_lock)
print('Prediction lock:',prediction_lock['lock_id'])

## 6. Unlock clean truth for scoring and oracle context only

Nothing in this cell modifies observable decisions. Clean truth is now used to score the locked arms and calculate an oracle ceiling.

In [ ]:
require((OUT/'prediction_lock.json').exists(),'Prediction lock missing.')
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
clean_map=image_map(ROOT/'source'/'clean')

def printed_ink(clean):return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def classify_changed(candidate,baseline,observed,clean_ink,truth_uncertain,mask):
    changed=(candidate!=baseline)&(candidate==observed)&mask
    tp=changed&clean_ink&~truth_uncertain
    fp=changed&~clean_ink&~truth_uncertain
    tu=changed&truth_uncertain
    return changed,tp,fp,tu

arm_rows=[];pixel_context=[];oracle_rows=[]
for page_key,z in page_runtime.items():
    page=z['page'];require(page in clean_map,f'{page}: clean truth missing')
    clean=cv2.imread(str(clean_map[page]),0);require(clean is not None and clean.shape==z['observed'].shape,f'{page}: clean shape mismatch')
    ink=printed_ink(clean);k=np.ones((3,3),np.uint8);near1=cv2.dilate(ink.astype(np.uint8),k)>0;eroded=cv2.erode(ink.astype(np.uint8),k)>0;truth_uncertain=np.logical_xor(near1,eroded)
    # Oracle paper veto: revert C1 where no clean print is within 8 px. Development ceiling only.
    near8=cv2.dilate(ink.astype(np.uint8),disk(8))>0
    oracle_paper=z['ring2']&~near8
    oracle=z['C1'].copy();oracle[oracle_paper]=z['C0'][oracle_paper]
    cv2.imwrite(str(PAGES/page_key/'ORACLE_PAPER_VETO.png'),oracle)
    arms={'C1_RING2':z['C1'],'A1_PRINT_SUPPORTED_ONLY':z['A1'],'A2_PAPER_VETO':z['A2'],'A3_PAPER_VETO_PLUS_RING1_STRONG':z['A3'],'ORACLE_PAPER_VETO':oracle}
    n,labels,stats,_=cv2.connectedComponentsWithStats(z['mask'].astype(np.uint8),8)
    for arm,cand in arms.items():
        changed,tp,fp,tu=classify_changed(cand,z['C0'],z['observed'],ink,truth_uncertain,z['mask'])
        cats=0
        for cid in set(labels[changed].tolist())-{0}:
            cm=labels==cid;ct=int((tp&cm).sum());cf=int((fp&cm).sum())
            if cf>=50 or (cf/max(ct,1)>1 and cf>=20):cats+=1
        arm_rows.append({'page_key':page_key,'page':page,'source_group':z['source_group'],'arm':arm,'selected':int(changed.sum()),'true_print_recovered':int(tp.sum()),'annotation_retained':int(fp.sum()),'truth_uncertain_retained':int(tu.sum()),'utility':int(tp.sum()-5*fp.sum()-500*cats),'catastrophic_components':cats})
    # Score the locked observable context itself.
    context_masks={'PAPER_SUPPORTED':z['paper'],'PRINT_SUPPORTED':z['ring2']&z['print_support'],'UNRESOLVED':z['unresolved'],'A3_STRONG_RING1':z['strong_ring1']}
    for cls,m in context_masks.items():
        pixel_context.append({'page_key':page_key,'page':page,'source_group':z['source_group'],'context_class':cls,'pixels':int(m.sum()),'clean_print_pixels':int((m&ink&~truth_uncertain).sum()),'clean_annotation_nonprint_pixels':int((m&~ink&~truth_uncertain).sum()),'truth_uncertain_pixels':int((m&truth_uncertain).sum())})
    # Oracle distance bands for opportunity diagnosis.
    dist=cv2.distanceTransform((~ink).astype(np.uint8),cv2.DIST_L2,5)
    for radius in ORACLE_NEAR_PRINT_RADII:
        m=z['ring2']&(dist<=radius)
        oracle_rows.append({'page_key':page_key,'page':page,'source_group':z['source_group'],'radius':radius,'ring2_pixels':int(z['ring2'].sum()),'selected_near_print':int(m.sum()),'true_print':int((m&ink&~truth_uncertain).sum()),'annotation_nonprint':int((m&~ink&~truth_uncertain).sum()),'truth_uncertain':int((m&truth_uncertain).sum())})

page_results=pd.DataFrame(arm_rows);page_results.to_csv(TABLES/'results_by_page.csv',index=False);pd.DataFrame(pixel_context).to_csv(TABLES/'observable_context_truth_diagnostics.csv',index=False);pd.DataFrame(oracle_rows).to_csv(TABLES/'oracle_distance_band_diagnostics.csv',index=False)
print(page_results.groupby('arm')[['true_print_recovered','annotation_retained','utility','catastrophic_components']].sum())

## 7. Pooled arm summaries with page-clustered bootstrap

In [ ]:
def boot_arm(g):
    rng=np.random.default_rng(SEED);pages=g.page_key.to_numpy();idx=g.set_index('page_key');out=[]
    for _ in range(BOOTSTRAP_RESAMPLES):
        q=idx.loc[rng.choice(pages,size=len(pages),replace=True)];tp=float(q.true_print_recovered.sum());fp=float(q.annotation_retained.sum());cat=float(q.catastrophic_components.sum());out.append((tp-5*fp-500*cat,fp/max(tp,1),fp/max(tp+fp,1)))
    return np.asarray(out,float)

rows=[]
for arm,g in page_results.groupby('arm'):
    tp=int(g.true_print_recovered.sum());fp=int(g.annotation_retained.sum());tu=int(g.truth_uncertain_retained.sum());cat=int(g.catastrophic_components.sum());b=boot_arm(g)
    rows.append({'arm':arm,'pages':g.page_key.nunique(),'source_groups':g.source_group.nunique(),'selected':int(g.selected.sum()),'true_print_recovered':tp,'annotation_retained':fp,'truth_uncertain_retained':tu,'false_true_ratio':fp/max(tp,1),'false_preservation_rate':fp/max(tp+fp,1),'utility':tp-5*fp-500*cat,'catastrophic_components':cat,'benefiting_pages_ge10':int((g.true_print_recovered>=10).sum()),'benefiting_source_groups':int(g.loc[g.true_print_recovered>=10,'source_group'].nunique()),'bootstrap_utility_95_low':float(np.quantile(b[:,0],.025)),'bootstrap_utility_95_high':float(np.quantile(b[:,0],.975)),'bootstrap_false_true_95_high':float(np.quantile(b[:,1],.975)),'bootstrap_false_preservation_95_high':float(np.quantile(b[:,2],.975))})
summary=pd.DataFrame(rows);summary.to_csv(TABLES/'pooled_arm_summary.csv',index=False);print(summary.to_string(index=False))

## 8. Incremental comparison against C1

In [ ]:
c1=page_results[page_results.arm=='C1_RING2'].set_index('page_key')
comparisons=[]
for arm in ['A1_PRINT_SUPPORTED_ONLY','A2_PAPER_VETO','A3_PAPER_VETO_PLUS_RING1_STRONG','ORACLE_PAPER_VETO']:
    q=page_results[page_results.arm==arm].set_index('page_key').loc[c1.index]
    comparisons.append({'arm':arm,'delta_true_print':int(q.true_print_recovered.sum()-c1.true_print_recovered.sum()),'delta_annotation':int(q.annotation_retained.sum()-c1.annotation_retained.sum()),'delta_utility':int(q.utility.sum()-c1.utility.sum()),'delta_catastrophes':int(q.catastrophic_components.sum()-c1.catastrophic_components.sum()),'true_print_retention_vs_C1':float(q.true_print_recovered.sum()/max(c1.true_print_recovered.sum(),1)),'annotation_retention_vs_C1':float(q.annotation_retained.sum()/max(c1.annotation_retained.sum(),1)),'pages_improved_utility_vs_C1':int((q.utility>c1.utility).sum()),'pages_worse_utility_vs_C1':int((q.utility<c1.utility).sum())})
comparison=pd.DataFrame(comparisons);comparison.to_csv(TABLES/'incremental_vs_C1.csv',index=False);print(comparison.to_string(index=False))

## 9. Annotator/style and source-group stratification

In [ ]:
page_meta=metadata.rename(columns={'acquisition_notes':'annotator'}) if 'acquisition_notes' in metadata.columns else metadata.assign(annotator='UNKNOWN')
joined=page_results.merge(page_meta[['page']+[c for c in ['annotator'] if c in page_meta.columns]],on='page',how='left')
joined['annotator']=joined.annotator.fillna('UNKNOWN')
by_annotator=joined.groupby(['annotator','arm'],as_index=False).agg(pages=('page','nunique'),true_print_recovered=('true_print_recovered','sum'),annotation_retained=('annotation_retained','sum'),utility=('utility','sum'),catastrophic_components=('catastrophic_components','sum'))
by_annotator['false_true_ratio']=by_annotator.annotation_retained/np.maximum(by_annotator.true_print_recovered,1)
by_annotator.to_csv(TABLES/'results_by_annotator.csv',index=False)
by_group=page_results.groupby(['source_group','arm'],as_index=False).agg(pages=('page','nunique'),true_print_recovered=('true_print_recovered','sum'),annotation_retained=('annotation_retained','sum'),utility=('utility','sum'),catastrophic_components=('catastrophic_components','sum'))
by_group['false_true_ratio']=by_group.annotation_retained/np.maximum(by_group.true_print_recovered,1)
by_group.to_csv(TABLES/'results_by_source_group.csv',index=False)
print(by_annotator.to_string(index=False))

## 10. Observable-paper quality diagnostics

In [ ]:
ctx=pd.read_csv(TABLES/'observable_context_truth_diagnostics.csv')
ctx_summary=ctx.groupby('context_class',as_index=False).agg(pixels=('pixels','sum'),clean_print_pixels=('clean_print_pixels','sum'),clean_annotation_nonprint_pixels=('clean_annotation_nonprint_pixels','sum'),truth_uncertain_pixels=('truth_uncertain_pixels','sum'),pages=('page','nunique'),source_groups=('source_group','nunique'))
ctx_summary['print_fraction']=ctx_summary.clean_print_pixels/np.maximum(ctx_summary.clean_print_pixels+ctx_summary.clean_annotation_nonprint_pixels,1)
ctx_summary['annotation_fraction']=ctx_summary.clean_annotation_nonprint_pixels/np.maximum(ctx_summary.clean_print_pixels+ctx_summary.clean_annotation_nonprint_pixels,1)
ctx_summary.to_csv(TABLES/'observable_context_truth_summary.csv',index=False)
# Primary diagnostic: how much false annotation does the paper veto remove for each true-print pixel lost?
c1s=summary.set_index('arm').loc['C1_RING2'];a2s=summary.set_index('arm').loc['A2_PAPER_VETO']
quality={'c1_true_print':int(c1s.true_print_recovered),'c1_annotation':int(c1s.annotation_retained),'a2_true_print':int(a2s.true_print_recovered),'a2_annotation':int(a2s.annotation_retained),'true_print_lost':int(c1s.true_print_recovered-a2s.true_print_recovered),'annotation_removed':int(c1s.annotation_retained-a2s.annotation_retained),'annotation_removed_per_true_lost':float((c1s.annotation_retained-a2s.annotation_retained)/max(c1s.true_print_recovered-a2s.true_print_recovered,1)),'a2_true_retention':float(a2s.true_print_recovered/max(c1s.true_print_recovered,1)),'a2_annotation_retention':float(a2s.annotation_retained/max(c1s.annotation_retained,1))}
atomic_json(TABLES/'paper_veto_quality.json',quality)
print(ctx_summary.to_string(index=False));print(json.dumps(quality,indent=2))

## 11. Fixed sensitivity table without choosing a post-hoc production threshold

In [ ]:
# This scans predeclared evidence variants for understanding only. A2_PRIMARY remains the named arm.
variants=[]
for band in SUPPORT_BANDS:
  for radius in [6,12,18]:
    for min_paper in [.60,.72,.82]:
      tp=fp=tu=selected=0
      for page_key,z in page_runtime.items():
        with np.load(MAPS/f'{page_key}.npz') as q:
          ring2=q['ring2_candidate']>0;normalised=q['normalised'];paper_fraction=q['paper_fraction'];dark_fraction=q['dark_fraction'];edge_fraction=q['edge_fraction'];dtd=q['distance_to_dark'];dte=q['distance_to_edge'];support_count=q['support_count']
        # support_count is primary-band identity. Rebuild alternate support map from observed/mask for band sensitivity.
        ctxv=observable_context(z['observed'],z['mask']);sp=ctxv['supports'][band];support=(sp['horizontal']|sp['vertical']|sp['compact']|sp['edge'])
        veto=ring2&(paper_fraction>=min_paper)&(dark_fraction<=PRIMARY_MAX_OUTSIDE_DARK_FRACTION)&(edge_fraction<=PRIMARY_MAX_OUTSIDE_EDGE_FRACTION)&(normalised>=PRIMARY_MIN_NORMALISED_INTENSITY)&(dtd>=PRIMARY_MIN_DISTANCE_TO_DARK)&(dte>=PRIMARY_MIN_DISTANCE_TO_EDGE)&~support
        cand=z['C1'].copy();cand[veto]=z['C0'][veto]
        clean=cv2.imread(str(clean_map[z['page']]),0);ink=printed_ink(clean);near=cv2.dilate(ink.astype(np.uint8),np.ones((3,3),np.uint8))>0;er=cv2.erode(ink.astype(np.uint8),np.ones((3,3),np.uint8))>0;unc=np.logical_xor(near,er)
        ch,t,f,u=classify_changed(cand,z['C0'],z['observed'],ink,unc,z['mask']);selected+=int(ch.sum());tp+=int(t.sum());fp+=int(f.sum());tu+=int(u.sum())
      variants.append({'support_band':band,'paper_radius_label':radius,'minimum_paper_fraction':min_paper,'selected':selected,'true_print':tp,'annotation':fp,'truth_uncertain':tu,'false_true_ratio':fp/max(tp,1),'utility_without_catastrophe_penalty':tp-5*fp})
sensitivity=pd.DataFrame(variants);sensitivity.to_csv(TABLES/'observable_policy_sensitivity.csv',index=False)
print('Sensitivity written. Primary policy remains unchanged and is not selected from this table.')

## 12. Visual outputs and focused review queues

In [ ]:
sns.set_theme(style='whitegrid')
# Arm totals.
order=['C1_RING2','A1_PRINT_SUPPORTED_ONLY','A2_PAPER_VETO','A3_PAPER_VETO_PLUS_RING1_STRONG','ORACLE_PAPER_VETO']
q=summary.set_index('arm').reindex(order).reset_index();x=np.arange(len(q))
fig,ax=plt.subplots(figsize=(13,6));ax.bar(x-.2,q.true_print_recovered,.4,label='true print recovered');ax.bar(x+.2,q.annotation_retained,.4,label='annotation retained');ax.set_xticks(x,[s.replace('_','\n') for s in q.arm]);ax.set_title('Notebook 77 arms: benefit and annotation cost');ax.legend();plt.tight_layout();plt.savefig(VISUAL/'arm_benefit_cost.png',dpi=180);plt.close(fig)
# Context truth composition.
q=ctx_summary.copy();bottom=np.zeros(len(q));plt.figure(figsize=(9,6))
for col,label in [('clean_print_pixels','clean print'),('clean_annotation_nonprint_pixels','annotation/non-print'),('truth_uncertain_pixels','truth uncertain')]:plt.bar(q.context_class,q[col],bottom=bottom,label=label);bottom+=q[col].to_numpy()
plt.xticks(rotation=15);plt.ylabel('Pixels');plt.title('What the observable context classes contain under clean truth');plt.legend();plt.tight_layout();plt.savefig(VISUAL/'observable_context_truth_composition.png',dpi=180);plt.close()
# Annotator A2 effect.
if len(by_annotator):
  a=by_annotator[by_annotator.arm.isin(['C1_RING2','A2_PAPER_VETO'])].copy();plt.figure(figsize=(10,6));sns.barplot(data=a,x='annotator',y='false_true_ratio',hue='arm');plt.title('C1 versus A2 trade-off by annotator/style');plt.tight_layout();plt.savefig(VISUAL/'annotator_tradeoff.png',dpi=180);plt.close()

# Review queue prioritises biggest C1 annotation reductions, then true-print losses.
piv=page_results.pivot(index=['page_key','page','source_group'],columns='arm',values=['true_print_recovered','annotation_retained','utility']).reset_index()
piv.columns=['_'.join([str(x) for x in c if str(x)!='']) for c in piv.columns]
piv['annotation_removed_by_A2']=piv['annotation_retained_C1_RING2']-piv['annotation_retained_A2_PAPER_VETO'];piv['true_print_lost_by_A2']=piv['true_print_recovered_C1_RING2']-piv['true_print_recovered_A2_PAPER_VETO'];piv['utility_delta_A2_vs_C1']=piv['utility_A2_PAPER_VETO']-piv['utility_C1_RING2'];piv=piv.sort_values(['annotation_removed_by_A2','true_print_lost_by_A2'],ascending=[False,False]);piv.to_csv(TABLES/'visual_review_queue.csv',index=False)

def sheet(rows,name,max_pages=12):
    rows=rows.head(max_pages)
    if len(rows)==0:return
    fig,axes=plt.subplots(len(rows),4,figsize=(18,max(5,5.5*len(rows))),squeeze=False)
    for i,(_,r) in enumerate(rows.iterrows()):
        d=PAGES/r.page_key;items=[('C0_INCUMBENT.png','C0'),('C1_RING2.png','C1'),('A2_PAPER_VETO.png','A2 paper veto'),('observable_context_overlay.png','context overlay')]
        for j,(fn,title) in enumerate(items):
            im=cv2.imread(str(d/fn),cv2.IMREAD_UNCHANGED)
            if im is not None and im.ndim==3:im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
            axes[i,j].imshow(im,cmap='gray' if im is not None and im.ndim==2 else None);axes[i,j].set_title(f'{r.page_key} {title}\nann removed={int(r.annotation_removed_by_A2)} true lost={int(r.true_print_lost_by_A2)}');axes[i,j].axis('off')
    plt.tight_layout();plt.savefig(VISUAL/f'{name}.png',dpi=130,bbox_inches='tight');plt.close(fig)
sheet(piv,'paper_veto_review_queue')
print('Visuals:',VISUAL)

## 13. Decision record

In [ ]:
a2=summary.set_index('arm').loc['A2_PAPER_VETO'];c1=summary.set_index('arm').loc['C1_RING2']
# Evidence flags, not production gates.
flags={
 'integrity_pass':bool(integrity[['A1_outside_drift','A2_outside_drift','A3_outside_drift','A2_non_veto_regression']].to_numpy().sum()==0),
 'a2_positive_utility':bool(a2.utility>0),
 'a2_bootstrap_utility_lower_positive':bool(a2.bootstrap_utility_95_low>0),
 'a2_retains_at_least_99_percent_c1_true_print':bool(a2.true_print_recovered>=.99*c1.true_print_recovered),
 'a2_removes_at_least_10_percent_c1_annotation':bool(a2.annotation_retained<=.90*c1.annotation_retained),
 'a2_no_more_catastrophes_than_c1':bool(a2.catastrophic_components<=c1.catastrophic_components),
}
if flags['integrity_pass'] and flags['a2_retains_at_least_99_percent_c1_true_print'] and flags['a2_removes_at_least_10_percent_c1_annotation'] and flags['a2_no_more_catastrophes_than_c1']:
    interpretation='Observable paper is a useful C1 veto candidate: A2 retained nearly all C1 print while removing a material share of annotation. Proceed to visual review and annotated-only shadow application before adoption.'
elif flags['integrity_pass'] and a2.utility>=c1.utility:
    interpretation='A2 improves or matches mathematical utility but does not meet the deliberately demanding 99%-print/10%-annotation heuristic. Treat as promising diagnostic evidence, not a routing decision.'
else:
    interpretation='The current observable-paper rule does not dominate C1. Retain C1 as the leading visual-restoration candidate and use Notebook 77 to diagnose whether paper evidence needs a learned or differently calibrated veto.'
record={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'notebook77-observable-paper-veto','development_only':True,'prediction_lock_id':prediction_lock['lock_id'],'primary_arm':'A2_PAPER_VETO','policy':POLICY,'flags':flags,'interpretation':interpretation,'operator_changed':False,'promotion_authority':False}
atomic_json(OUT/'notebook77_verdict.json',record)
lines=['NOTEBOOK 77 — OBSERVABLE-PAPER VETO FOR RING-2 PRESERVATION','',f'Source Notebook 75: {NB75}',f'Pages: {len(page_runtime)}',f'Prediction lock: {prediction_lock["lock_id"]}','',summary.to_string(index=False),'','PAPER-VETO QUALITY',json.dumps(quality,indent=2),'','EVIDENCE FLAGS']+[f'- {k}: {v}' for k,v in flags.items()]+['','INTERPRETATION',interpretation,'','This is development evidence only. No incumbent/operator/production output was changed.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines));print('\n'.join(lines))

## 14. Standard final upload package

In [ ]:
artefacts=[OUT/'executive_summary.txt',OUT/'notebook77_verdict.json',OUT/'prediction_lock.json',INPUTS/'predeclared_policy.json',AUDIT/'source_record.json',AUDIT/'observable_arm_integrity.csv',AUDIT/'page_output_hashes.csv',TABLES/'pooled_arm_summary.csv',TABLES/'incremental_vs_C1.csv',TABLES/'results_by_page.csv',TABLES/'results_by_source_group.csv',TABLES/'results_by_annotator.csv',TABLES/'observable_context_truth_summary.csv',TABLES/'paper_veto_quality.json',TABLES/'oracle_distance_band_diagnostics.csv',TABLES/'observable_policy_sensitivity.csv',TABLES/'visual_review_queue.csv',VISUAL/'arm_benefit_cost.png',VISUAL/'observable_context_truth_composition.png',VISUAL/'paper_veto_review_queue.png']
if (VISUAL/'annotator_tradeoff.png').exists():artefacts.append(VISUAL/'annotator_tradeoff.png')
for p in artefacts:
    require(p.exists(),f'Missing required upload artefact: {p}');shutil.copy2(p,UPLOAD/p.name)
(UPLOAD/'README.txt').write_text('Notebook 77 tests observable paper as a veto on C1 Ring-2 preservation. Start with executive_summary.txt, pooled_arm_summary.csv, incremental_vs_C1.csv, paper_veto_quality.json, observable_context_truth_summary.csv and paper_veto_review_queue.png. ORACLE results are development ceilings and are not deployable.\n')
manifest_rows=[]
for p in sorted(UPLOAD.iterdir()):
    if p.is_file():manifest_rows.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest_rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text('\n'.join(sorted(p.name for p in UPLOAD.iterdir() if p.is_file())))
zip_path=UPLOAD/'notebook77_observable_paper_veto_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(UPLOAD.iterdir()):
        if p.is_file() and p!=zip_path:z.write(p,p.name)
print('Final bundle:',zip_path)

## Reading Notebook 77

The critical comparisons are:

1. **A2 versus C1:** how much C1 annotation is removed for each C1 true-print pixel lost?
2. **Observable context truth summary:** does `PAPER_SUPPORTED` actually contain mostly annotation/non-print, while `PRINT_SUPPORTED` and `UNRESOLVED` contain most of the useful print?
3. **A2 versus oracle:** how much of the theoretically available paper veto is captured by observable evidence?
4. **Annotator/style stratification:** does A2 narrow the David/Ali/Christian difference without sacrificing the C1 benefit?
5. **A3 versus A2:** is there any extremely narrow, strongly supported Ring-1 subset worth retaining for later review, without recreating blanket C2 behaviour?

A2 is intentionally conservative in only one direction: it reverts C1 **only** when paper is positively supported. Lack of print support is not enough by itself to revert a pixel, because masked print may have no visible continuation.

The primary policy is declared before scoring. The sensitivity table is diagnostic and must not be used to silently choose a new production rule from the same outcomes.